## 1. Load Dataset

In [16]:
from google.colab import files
import pandas as pd

# Upload file tourism_with_id.csv dari komputer kamu
uploaded = files.upload()

# Load dataset
df_places = pd.read_csv('tourism_with_id.csv')
print("Dataset berhasil diisi! Jumlah data awal:", df_places.shape)

Saving tourism_with_id.csv to tourism_with_id (2).csv
Dataset berhasil diisi! Jumlah data awal: (437, 13)


## 2. Cleaning & Preprocessing

In [17]:
import numpy as np

def process_raw_data(df):
    print("=== 1. DATA CLEANING ===")

    # Hapus kolom tidak relevan/sampah
    cols_to_drop = [col for col in ['Unnamed: 11', 'Unnamed: 12'] if col in df.columns]
    df_clean = df.drop(columns=cols_to_drop)
    print(f"-> Kolom sampah dihapus: {cols_to_drop}")

    # Tangani Missing Values di kolom Time_Minutes
    if 'Time_Minutes' in df_clean.columns:
        median_val = df_clean['Time_Minutes'].median()
        df_clean['Time_Minutes'] = df_clean['Time_Minutes'].fillna(median_val)
        print(f"-> Missing values 'Time_Minutes' diisi dengan median: {median_val}")

    print("=== 2. DATA PREPROCESSING ===")

    # Gabungkan teks (Category, City, Description) menjadi satu fitur teks tunggal
    df_clean['Content_Feature'] = (
        df_clean['Category'].astype(str) + " " +
        df_clean['City'].astype(str) + " " +
        df_clean['Description'].astype(str)
    )
    print("-> Fitur teksgabungan ('Content_Feature') berhasil dibuat!")

    return df_clean

# Jalankan fungsi
df_places_clean = process_raw_data(df_places)
df_places_clean.head(3)

=== 1. DATA CLEANING ===
-> Kolom sampah dihapus: ['Unnamed: 11', 'Unnamed: 12']
-> Missing values 'Time_Minutes' diisi dengan median: 60.0
=== 2. DATA PREPROCESSING ===
-> Fitur teksgabungan ('Content_Feature') berhasil dibuat!


,Place_Id,Place_Name,Description,Category,City,Price,Rating,Time_Minutes,Coordinate,Lat,Long,Content_Feature
0,1,Monumen Nasional,Monumen Nasional atau yang populer disingkat d...,Budaya,Jakarta,20000,4.6,15.0,"{'lat': -6.1753924, 'lng': 106.8271528}",-6.175392,106.827153,Budaya Jakarta Monumen Nasional atau yang popu...
1,2,Kota Tua,"Kota tua di Jakarta, yang juga bernama Kota Tu...",Budaya,Jakarta,0,4.6,90.0,"{'lat': -6.137644799999999, 'lng': 106.8171245}",-6.137645,106.817125,"Budaya Jakarta Kota tua di Jakarta, yang juga ..."
2,3,Dunia Fantasi,Dunia Fantasi atau disebut juga Dufan adalah t...,Taman Hiburan,Jakarta,270000,4.6,360.0,"{'lat': -6.125312399999999, 'lng': 106.8335377}",-6.125312,106.833538,Taman Hiburan Jakarta Dunia Fantasi atau diseb...


## 3. Matriks ML

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("=== 3. PENYIAPAN MATRIKS INTERAKSI / SIMILARITY ===")

# 1. TF-IDF Vectorization pada teks deskripsi tempat wisata
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df_places_clean['Content_Feature'])

# 2. Pembuatan Cosine Similarity Matrix (Matriks Matriks Interaksi Tempat vs Tempat)
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

# 3. Ubah ke DataFrame agar mudah diindeks berdasarkan Place_Name / Place_Id
sim_df = pd.DataFrame(
    cosine_sim_matrix,
    index=df_places_clean['Place_Name'],
    columns=df_places_clean['Place_Name']
)

print(f"-> Ukuran TF-IDF Matrix (Tempat x Kata Unik) : {tfidf_matrix.shape}")
print(f"-> Ukuran Matriks Interaksi Cosine Similarity : {cosine_sim_matrix.shape}")
print("=== PENGOLAHAN DATA Selesai! MATRIKS SIAP DIGUNAKAN UNTUK REKOMENDASI ===")

# Tampilkan preview matriks kemiripan antar tempat wisata
sim_df.iloc[:5, :5]

=== 3. PENYIAPAN MATRIKS INTERAKSI / SIMILARITY ===
-> Ukuran TF-IDF Matrix (Tempat x Kata Unik) : (437, 6526)
-> Ukuran Matriks Interaksi Cosine Similarity : (437, 437)
=== PENGOLAHAN DATA Selesai! MATRIKS SIAP DIGUNAKAN UNTUK REKOMENDASI ===


Place_Name,Monumen Nasional,Kota Tua,Dunia Fantasi,Taman Mini Indonesia Indah (TMII),Atlantis Water Adventure
Place_Name,,,,,
Monumen Nasional,1.000000,0.068994,0.114829,0.062871,0.037908
Kota Tua,0.068994,1.000000,0.064035,0.052301,0.052207
Dunia Fantasi,0.114829,0.064035,1.000000,0.153654,0.119051
Taman Mini Indonesia Indah (TMII),0.062871,0.052301,0.153654,1.000000,0.069414
Atlantis Water Adventure,0.037908,0.052207,0.119051,0.069414,1.000000
